# Notebook 4: Benchmark Evaluation & Majority@8 Self-Consistency (Kaggle)
**Project**: Autonomous Text-to-SQL Generator Aligned via RLVR + GRPO  
**Hardware**: Kaggle GPU (T4 x 1)  
**Phase**: Phase 5 — AST Security Hardening, Majority@8 Voting & Confidence Handling, Final Benchmarking

## Audit Corrections Applied (v2)
| # | Issue | Correction |
|---|-------|------------|
| 1 | GRPO adapter loading was optional (fell back to SFT/base silently) | **GRPO adapter is now mandatory. Notebook aborts if unavailable.** Adapter path and SHA-256 recorded in every result row. |
| 2 | Tie-handling described as "greedy fallback" but was actually first-seen cluster selection | Corrected: tie returns the **first-seen** tied cluster's representative SQL. Status `NO_CONSENSUS_TIE` is correctly labelled. |
| 3 | `is_ast_safe` skipped table allowlist when `valid_tables` was empty | Table allowlist enforced unconditionally; empty list raises `ValueError`. Column validation absent — **documented explicitly**. |
| 4 | Notebook 4 had its own diverged `parse_sql` (single return value, no trailing-text check) | Replaced with the **strict two-value `parse_sql`** shared with Notebook 3 (counts tags, checks trailing text). |
| 5 | Sandbox lacked path validation; sandbox in NB4 diverged from NB3 | Reuses the **exact sandbox** from NB3: path existence check, read-only URI, `PRAGMA query_only`, authorizer, progress-handler deadline, 5 001-row cap. |
| 6 | Only `greedy_correct` / `majority8_correct` flags recorded; no validity metrics | Added `greedy_sql_valid` (format+AST+exec), `maj8_sql_valid`, `greedy_exec_ok`, `maj8_exec_ok` per row and aggregated %. |
| 7 | Majority@8 latency not measured; only greedy latency stored | Added `maj8_latency_ms` measured wall-clock from first sample call to cluster selection. |
| 8 | `np.percentile` used without documenting interpolation method | Percentiles now use `method='linear'` (NumPy default, documented inline). |
| 9 | JSON records lacked SQL text, candidate statuses, cluster sizes, adapter checksum | Each record now stores `greedy_sql`, `maj8_sql`, `maj8_cluster_sizes`, `maj8_n_valid_candidates`, and `adapter_sha256`. |
| 10 | Eval set hardcoded to `eval_data[:50]` (undocumented first-50 selection) | Labelled as `EVAL_MODE = "first_50"`. Set `EVAL_MODE = "full"` to evaluate all 1 034 Spider dev examples. |

---

In [ ]:
# ── Cell 1: Install Pinned Dependencies ───────────────────────────────────
!pip uninstall -y -q torchvision
!pip install -q \
    "transformers==4.57.3" \
    "trl==0.25.0" \
    "peft==0.18.1" \
    "datasets>=3.0.0,<5.0.0" \
    "accelerate>=1.2.1,<2.0.0" \
    "bitsandbytes>=0.45.2" \
    "sqlglot>=26.0.0" \
    "pandas" \
    "tabulate"
print("Evaluation packages ready.")

In [ ]:
# ── Cell 2: Paths, Environment & Dataset/Adapter Unpacking ────────────────
#
# AUDIT FIX #10: EVAL_MODE controls evaluation scope.
#   "first_50"  — first 50 dev examples (fast, used for prior runs)
#   "full"      — all 1 034 Spider dev examples (full benchmark)
#
# AUDIT FIX #1: GRPO adapter is now mandatory. The notebook aborts
#   (raises RuntimeError) if the GRPO adapter is unavailable.
#   SFT adapter loading is kept for reference info only.
# ─────────────────────────────────────────────────────────────────────────

EVAL_MODE = "first_50"  # Change to "full" for complete 1 034-example evaluation

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import hashlib
import json
import shutil
import zipfile
import torch
from pathlib import Path

assert torch.cuda.is_available(), "CUDA GPU required for evaluation!"
major, minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)} (Compute Capability {major}.{minor})")

WORKING_DIR = Path("/kaggle/working")
DATA_DIR    = WORKING_DIR / "data"
REPORTS_DIR = WORKING_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset unpack ────────────────────────────────────────────────────────
dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"

def setup_dataset():
    if dev_manifest.exists():
        return
    all_zips = list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))
    for z in all_zips:
        if any(k in z.name.lower() for k in ["sql_grpo", "data", "processed", "spider"]):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    names = zf.namelist()
                    if any("dev_manifest.jsonl" in n for n in names):
                        print(f"📦 Unpacking dataset from: {z} ...")
                        has_data_root = any(n.startswith("data/") for n in names)
                        extract_target = WORKING_DIR if has_data_root else DATA_DIR
                        zf.extractall(extract_target)
                        break
            except Exception as e:
                print(f"Warning inspecting {z}: {e}")
    if not dev_manifest.exists():
        found = list(Path("/kaggle/input").glob("**/dev_manifest.jsonl"))
        if found:
            dataset_root = found[0].parent.parent if found[0].parent.name == "processed" else found[0].parent
            src_proc = dataset_root / "processed"
            dst_proc = DATA_DIR / "processed"
            if src_proc.exists() and not dst_proc.exists():
                shutil.copytree(src_proc, dst_proc)
            src_raw = dataset_root / "raw"
            dst_raw = DATA_DIR / "raw"
            if src_raw.exists() and not dst_raw.exists():
                shutil.copytree(src_raw, dst_raw)

setup_dataset()
assert dev_manifest.exists(), "❌ dev_manifest.jsonl missing! Please attach sql_grpo_data.zip."

# ── Adapter unpack ────────────────────────────────────────────────────────
GRPO_ADAPTER = WORKING_DIR / "outputs" / "grpo" / "final_adapter"
SFT_ADAPTER  = WORKING_DIR / "outputs" / "sft"  / "best_adapter"

def setup_adapters():
    all_zips = list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))
    for z in all_zips:
        z_lower = z.name.lower()
        if any(k in z_lower for k in ["rlvr", "grpo", "sft"]):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    if any("adapter_model.safetensors" in n for n in zf.namelist()):
                        print(f"📦 Unpacking adapter archive: {z.name} ...")
                        zf.extractall(WORKING_DIR)
            except Exception as e:
                print(f"Warning inspecting {z}: {e}")

    for adapter_path, label, search_patterns in [
        (GRPO_ADAPTER, "GRPO", ["**/final_adapter/adapter_model.safetensors",
                                 "**/*rlvr*/adapter_model.safetensors",
                                 "**/*grpo*/adapter_model.safetensors"]),
        (SFT_ADAPTER,  "SFT",  ["**/best_adapter/adapter_model.safetensors",
                                 "**/*sft*/adapter_model.safetensors"]),
    ]:
        if not (adapter_path / "adapter_model.safetensors").exists():
            found = []
            for pat in search_patterns:
                found += list(Path("/kaggle/input").glob(pat))
                found += list(WORKING_DIR.glob(pat))
            candidates = [p for p in found if p.parent != adapter_path]
            if candidates:
                src = candidates[0].parent
                print(f"📁 Copying {label} adapter from {src} -> {adapter_path} ...")
                adapter_path.mkdir(parents=True, exist_ok=True)
                for f in src.glob("*"):
                    if f.is_file():
                        shutil.copy2(f, adapter_path / f.name)

setup_adapters()

# ── AUDIT FIX #1: GRPO adapter is mandatory — abort if missing ───────────
grpo_weights = GRPO_ADAPTER / "adapter_model.safetensors"
if not grpo_weights.exists():
    raise RuntimeError(
        "❌ GRPO adapter not found at:\n"
        f"   {GRPO_ADAPTER}\n"
        "Please attach sql_grpo_rlvr.zip to this Kaggle notebook as an input dataset."
    )

# Compute and record GRPO adapter SHA-256 at load time
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest().upper()

ADAPTER_SHA256 = sha256_file(grpo_weights)
print(f"✅ GRPO adapter found: {GRPO_ADAPTER}")
print(f"   SHA-256: {ADAPTER_SHA256}")
print(f"   SFT adapter present: {(SFT_ADAPTER / 'adapter_model.safetensors').exists()}")
print(f"   Eval mode: {EVAL_MODE}")

In [ ]:
# Hardened Shared SQL Parser, AST Policy, Authorizer, Canonical Comparator & Sandbox
import hashlib
import json as _json
import math
import sqlite3
import time
from collections import Counter
from decimal import Decimal
from typing import Any, Dict, List, Optional, Tuple
import sqlglot
from sqlglot import exp

SAFE_FUNCTIONS = {
    "count", "sum", "avg", "min", "max", "total", "group_concat", "string_agg",
    "lower", "upper", "length", "octet_length", "substr", "substring",
    "trim", "ltrim", "rtrim", "replace", "instr", "hex", "unicode", "char", "quote",
    "round", "abs", "ceil", "ceiling", "floor", "mod", "power", "sqrt", "sign",
    "coalesce", "nullif", "ifnull", "iif",
    "strftime", "date", "time", "datetime", "julianday", "unixepoch",
    "cast", "like", "glob", "typeof",
    "rank", "dense_rank", "row_number", "ntile", "lag", "lead", "first_value", "last_value",
}
SYSTEM_TABLES = {
    "sqlite_master", "sqlite_schema", "sqlite_temp_master",
    "sqlite_temp_schema", "sqlite_sequence", "sqlite_stat1", "sqlite_stat4",
}

def extract_completion_text(c: Any) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        for msg in reversed(c):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
            if isinstance(msg, dict) and "content" in msg:
                return msg.get("content", "")
        if len(c) > 0 and isinstance(c[-1], str):
            return c[-1]
        return str(c)
    if isinstance(c, dict):
        return c.get("content", str(c))
    return str(c)

def parse_sql(text: str) -> Tuple[Optional[str], bool]:
    if not text or "<sql>" not in text or "</sql>" not in text:
        return None, False
    if text.count("<sql>") != 1 or text.count("</sql>") != 1:
        return None, False
    idx1 = text.find("<sql>") + len("<sql>")
    idx2 = text.find("</sql>")
    if idx1 >= idx2:
        return None, False
    if text[idx2 + len("</sql>"):].strip():
        return None, False
    sql = text[idx1:idx2].strip()
    if sql.startswith("```"):
        lines = sql.splitlines()[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        sql = chr(10).join(lines).strip()
    return (sql, True) if sql else (None, False)

def is_ast_safe(sql: str, valid_tables: Optional[List[str]] = None) -> bool:
    if valid_tables is not None and len(valid_tables) == 0:
        raise ValueError("is_ast_safe: valid_tables cannot be an empty list.")
    try:
        stmts = sqlglot.parse(sql, read="sqlite")
        if not stmts or len(stmts) != 1 or stmts[0] is None:
            return False
        ast = stmts[0]
        if not isinstance(ast, (exp.Select, exp.Union, exp.Intersect, exp.Except)):
            return False
        for with_node in ast.find_all(exp.With):
            if with_node.args.get("recursive"):
                return False
        for sel in ast.find_all(exp.Select):
            if not sel.expressions:
                return False
        prohibited = tuple(
            t for t in (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop,
                        getattr(exp, "AlterTable", getattr(exp, "Alter", None)),
                        exp.Pragma, exp.Command, exp.Transaction,
                        getattr(exp, "Attach", None), getattr(exp, "Detach", None),
                        getattr(exp, "Analyze", None)) if t is not None
        )
        if any(ast.find_all(prohibited)):
            return False
        for node in ast.walk():
            if isinstance(node, exp.Anonymous):
                if (node.name or "").lower() not in SAFE_FUNCTIONS:
                    return False
        cte_names = {c.alias.lower() for c in ast.find_all(exp.CTE) if c.alias}
        phys_tables = set()
        for t in ast.find_all(exp.Table):
            if isinstance(t.this, (exp.Anonymous, exp.Func)):
                return False
            if (t.catalog or "") or (t.db or "").lower() not in {"", "main"}:
                return False
            tn = t.name.lower() if t.name else ""
            if not tn or tn in cte_names:
                continue
            if tn in SYSTEM_TABLES or tn.startswith("sqlite_") or tn.startswith("pragma_"):
                return False
            phys_tables.add(tn)
        if valid_tables is not None:
            allowed = {str(t).lower() for t in valid_tables}
            if phys_tables - allowed:
                return False
        return True
    except Exception:
        return False

def create_authorizer(allowed_tables: Optional[List[str]] = None):
    norm_tbls = {t.lower() for t in allowed_tables} if allowed_tables is not None else None
    def callback(action, arg1, arg2, db, trigger):
        if action == sqlite3.SQLITE_SELECT:
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_READ:
            tname = (arg1 or "").lower()
            dname = (db or "").lower()
            if dname not in {"", "main"}:
                return sqlite3.SQLITE_DENY
            if tname in SYSTEM_TABLES or tname.startswith("sqlite_") or tname.startswith("pragma_"):
                return sqlite3.SQLITE_DENY
            if norm_tbls is not None and tname not in norm_tbls:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_FUNCTION:
            if (arg2 or "").lower() in SAFE_FUNCTIONS:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    return callback

def _canonical_val(val: Any):
    if val is None:
        return ("__NULL__", None)
    if isinstance(val, bool):
        return ("bool", val)
    if isinstance(val, int):
        return ("int", val)
    if isinstance(val, float):
        if math.isnan(val):
            return ("float_nan", "nan")
        if math.isinf(val):
            return ("float_inf", 1 if val > 0 else -1)
        return ("float", 0.0 if val == 0.0 else float(val))
    if isinstance(val, Decimal):
        return ("decimal", str(val.normalize()))
    if isinstance(val, (bytes, bytearray, memoryview)):
        return ("bytes", bytes(val).hex())
    return ("str", str(val))

def canonical_row(row: Tuple[Any, ...]):
    return tuple(_canonical_val(v) for v in row)

def compare_multiset_rows(cand_rows: List[Tuple[Any, ...]], gold_rows: List[Tuple[Any, ...]]) -> bool:
    return Counter(canonical_row(r) for r in cand_rows) == Counter(canonical_row(r) for r in gold_rows)

def execute_sandbox_query(
    db_path: "Path",
    sql: str,
    timeout_sec: float = 2.0,
    row_limit: int = 5000,
    byte_limit: int = 1_048_576,
    allowed_tables: Optional[List[str]] = None,
) -> Tuple[bool, Optional[List[Tuple]], bool, float, Optional[str]]:
    start = time.perf_counter()
    deadline = start + timeout_sec
    conn = None
    cur = None
    try:
        resolved = db_path.resolve(strict=True)
        uri = resolved.as_uri() + "?mode=ro"
        conn = sqlite3.connect(uri, uri=True, timeout=1.0)
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(create_authorizer(allowed_tables=allowed_tables))
        conn.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10_000)
        cur = conn.cursor()
        cur.execute(sql)
        rows = []
        total_bytes = 0
        truncated = False
        while True:
            rem = (row_limit + 1) - len(rows)
            if rem <= 0:
                truncated = True
                break
            batch = cur.fetchmany(min(256, rem))
            if not batch:
                break
            for r in batch:
                rows.append(r)
                total_bytes += sum(
                    len(v) if isinstance(v, (bytes, bytearray))
                    else (len(v.encode("utf-8", "replace")) if isinstance(v, str) else 8)
                    for v in r
                )
                if len(rows) > row_limit or total_bytes > byte_limit:
                    truncated = True
                    break
            if truncated:
                break
        dur_ms = (time.perf_counter() - start) * 1000.0
        if truncated:
            return False, None, True, dur_ms, "RESULT_TRUNCATED"
        return True, rows, False, dur_ms, None
    except Exception as e:
        dur_ms = (time.perf_counter() - start) * 1000.0
        return False, None, False, dur_ms, str(e)
    finally:
        if cur:
            try:
                cur.close()
            except Exception:
                pass
        if conn:
            try:
                conn.set_progress_handler(None, 0)
                conn.set_authorizer(None)
                conn.close()
            except Exception:
                pass


import sqlite3 as _sqlite3_check
from collections import defaultdict
_sample_dbs = list((DATA_DIR / "raw" / "spider" / "database").glob("*/*.sqlite"))
assert _sample_dbs, "No Spider SQLite databases found under data/raw/spider/database/"
_test_db = _sample_dbs[0]
with _sqlite3_check.connect(_test_db) as _c:
    _tbls = [r[0] for r in _c.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
_test_tbl = _tbls[0] if _tbls else None
assert parse_sql(f"<sql>SELECT * FROM {_test_tbl} LIMIT 1</sql>") == (f"SELECT * FROM {_test_tbl} LIMIT 1", True)
assert parse_sql("<sql>SELECT 1</sql> trailing") == (None, False)
if _test_tbl:
    assert is_ast_safe(f"SELECT * FROM {_test_tbl} LIMIT 1", [_test_tbl]) is True
    assert is_ast_safe("SELECT * FROM sqlite_master", [_test_tbl]) is False
    assert is_ast_safe("SELECT * FROM pragma_table_info('x')", [_test_tbl]) is False
    assert is_ast_safe("SELECT * FROM fake_xyz_nonexistent", [_test_tbl]) is False
    try:
        is_ast_safe("SELECT 1", [])
        assert False, "Expected ValueError for empty valid_tables"
    except ValueError:
        pass
    _ok, _rows, _trunc, _dur, _err = execute_sandbox_query(_test_db, f"SELECT * FROM {_test_tbl} LIMIT 3")
    assert _ok and not _trunc and _rows is not None
    _ok2, _, _, _, _ = execute_sandbox_query(_test_db, f"INSERT INTO {_test_tbl} DEFAULT VALUES")
    assert not _ok2
print("✅ Shared hardened parser, AST verifier, canonical comparator, and sandbox verified.")

In [ ]:
# ── Cell 4: Load Base Model + Mandatory GRPO Adapter ──────────────────────
#
# AUDIT FIX #1 (enforced): Only the GRPO adapter is loaded here.
# The earlier RuntimeError in Cell 2 already guarantees the GRPO adapter
# weights exist before this cell runs. No fallback to SFT or base.
# ─────────────────────────────────────────────────────────────────────────
import transformers.utils.import_utils as _imp
_imp.is_torchvision_available = lambda: False

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Required for batch left-padded generation

major, minor = torch.cuda.get_device_capability()
supports_bf16 = (major >= 8)
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f"Compute dtype: {'bfloat16' if supports_bf16 else 'float16'} (CC {major}.{minor})")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)

# GRPO adapter loading is mandatory — no conditional branch
model = PeftModel.from_pretrained(base_model, str(GRPO_ADAPTER))
model.eval()

print(f"🚀 Evaluating GRPO-Aligned Policy: {GRPO_ADAPTER}")
print(f"   Adapter SHA-256 : {ADAPTER_SHA256}")
print(f"   Model device    : {model.device}")
print(f"   Peak VRAM       : {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")

In [ ]:
# ── Cell 5: Majority@8 Consensus Voting Engine ────────────────────────────
#
# AUDIT FIX #2 (tie handling): When two clusters share the largest size,
# the function returns the **first-seen** tied cluster's representative SQL
# (i.e. the cluster that appeared first in iteration order over the clusters
# dict). This is NOT a greedy fallback — it is first-seen selection.
# The status is correctly returned as "NO_CONSENSUS_TIE".
#
# AUDIT FIX #7: Majority@8 wall-clock latency is now measured end-to-end
# (from first stochastic generation call to cluster selection completion).
#
# AUDIT FIX #9: Returns cluster_sizes list and n_valid_candidates count.
# ─────────────────────────────────────────────────────────────────────────

def majority_8_inference(
    prompt_messages: list,
    db_file: "Path",
    valid_tables: List[str],
    k: int = 8,
) -> Tuple[Optional[str], Optional[List[Tuple]], str, List[int], int, float]:
    """
    Run Majority@8 stochastic generation and plurality voting.

    Returns:
        selected_sql     : SQL string of the elected candidate (None on failure).
        selected_rows    : Execution result rows of the elected candidate.
        status           : One of CONSENSUS_ELECTED / LOW_CONFIDENCE /
                           NO_CONSENSUS_TIE / ALL_FAILED.
        cluster_sizes    : Sorted descending list of valid cluster sizes.
        n_valid_candidates: Number of candidates that parsed and executed successfully.
        latency_ms       : End-to-end wall-clock latency in milliseconds.

    Tie-breaking note (AUDIT FIX #2):
        When the two largest clusters have equal size, the function selects the
        **first-seen** tied cluster (dict insertion order) and returns status
        "NO_CONSENSUS_TIE". This is first-seen selection, NOT greedy decoding.
    """
    t_start = time.perf_counter()

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            num_return_sequences=k,
            pad_token_id=tokenizer.pad_token_id,
        )

    in_len = inputs.input_ids.shape[1]

    # ── Cluster candidates by multiset execution-result hash ─────────────
    # clusters: dict[signature -> [(sql, rows), ...]]
    # Maintains insertion order (Python 3.7+) which determines first-seen
    # cluster for tie-breaking.
    clusters: dict = {}
    n_valid = 0
    for i in range(k):
        raw = tokenizer.decode(outputs[i][in_len:], skip_special_tokens=True)
        sql, fmt_ok = parse_sql(raw)
        if not fmt_ok or sql is None:
            continue
        # AUDIT FIX #3: valid_tables is always non-empty (enforced by is_ast_safe)
        if not is_ast_safe(sql, valid_tables):
            continue
        ok, rows, trunc, _, _ = execute_sandbox_query(db_file, sql, timeout_sec=2.0, allowed_tables=valid_tables)
        if not ok or trunc or rows is None:
            continue
        n_valid += 1
        # Canonical multiset signature: sort by string representation for stability
        sig = tuple(sorted((canonical_row(r) for r in rows), key=repr))
        if sig not in clusters:
            clusters[sig] = []
        clusters[sig].append((sql, rows))

    # ── Determine voting outcome ─────────────────────────────────────────
    latency_ms = (time.perf_counter() - t_start) * 1000.0

    if not clusters:
        return None, None, "ALL_FAILED", [], n_valid, latency_ms

    # Sort clusters by size descending; ties broken by first-seen order
    # (sorted is stable, so equal-size clusters preserve insertion order)
    sorted_clusters = sorted(clusters.values(), key=len, reverse=True)
    cluster_sizes = [len(c) for c in sorted_clusters]
    top_cluster = sorted_clusters[0]
    representative_sql, representative_rows = top_cluster[0]  # first entry in top cluster

    # AUDIT FIX #2: Tie detection — second cluster same size as first
    if len(sorted_clusters) > 1 and len(sorted_clusters[1]) == len(top_cluster):
        # Tie: return first-seen cluster's representative. NOT a greedy decode.
        return representative_sql, representative_rows, "NO_CONSENSUS_TIE", cluster_sizes, n_valid, latency_ms

    # Low confidence: winning cluster has fewer than 3 supporting candidates
    if len(top_cluster) < 3:
        return representative_sql, representative_rows, "LOW_CONFIDENCE", cluster_sizes, n_valid, latency_ms

    return representative_sql, representative_rows, "CONSENSUS_ELECTED", cluster_sizes, n_valid, latency_ms

print("✅ Majority@8 voting engine initialized.")
print("   Tie-breaking: first-seen cluster selection (NOT greedy decode).")
print("   Table allowlist: enforced unconditionally via is_ast_safe.")
print("   Column validation: NOT performed (SQLite authorizer enforces at exec time).")

In [ ]:
# ── Cell 6: Benchmark Execution — Greedy Pass@1 vs Majority@8 ─────────────
#
# AUDIT FIX #6:  greedy_sql_valid, maj8_sql_valid, greedy_exec_ok,
#                maj8_exec_ok added per record and aggregated.
# AUDIT FIX #7:  maj8_latency_ms stored separately from greedy_latency_ms.
# AUDIT FIX #8:  Percentiles use numpy method='linear' (default), documented.
# AUDIT FIX #9:  greedy_sql, maj8_sql, maj8_cluster_sizes,
#                maj8_n_valid_candidates, adapter_sha256 saved per record.
# AUDIT FIX #10: EVAL_MODE selects first-50 or full-1034 dev set.
# ─────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

assert dev_manifest.exists(), "dev_manifest.jsonl missing!"
eval_data = []
with open(dev_manifest, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_data.append(json.loads(line))

# AUDIT FIX #10: Evaluation scope selection
if EVAL_MODE == "full":
    eval_subset = eval_data  # All 1 034 Spider dev examples
    print(f"EVAL_MODE=full: Benchmarking all {len(eval_subset)} Spider dev examples.")
else:
    # "first_50": first 50 dev examples by manifest order (consistent with prior runs)
    EVAL_SIZE = min(50, len(eval_data))
    eval_subset = eval_data[:EVAL_SIZE]
    print(f"EVAL_MODE=first_50: Benchmarking first {len(eval_subset)} dev examples.")
    print("  (Set EVAL_MODE = 'full' in Cell 2 for the complete 1 034-example Spider benchmark.)")

print(f"\nRunning Greedy Pass@1 and Majority@8 evaluation...")
print(f"Adapter SHA-256: {ADAPTER_SHA256}")

results = []

for idx, row in enumerate(eval_subset):
    db_id    = row["db_id"]
    db_file  = DATA_DIR / "raw" / "spider" / "database" / db_id / f"{db_id}.sqlite"
    gold_raw = json.loads(row["gold_result_json"])
    gold_rows = [tuple(x) if isinstance(x, list) else (x,) for x in gold_raw]
    valid_tables = row["valid_tables"]

    # ── 1. Greedy Pass@1 ─────────────────────────────────────────────────
    p_text = tokenizer.apply_chat_template(
        row["prompt"], tokenize=False, add_generation_prompt=True
    )
    inp = tokenizer(p_text, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        g_out = model.generate(
            **inp,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    g_lat = (time.perf_counter() - t0) * 1000.0
    g_comp = tokenizer.decode(g_out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

    g_sql, g_fmt_ok = parse_sql(g_comp)  # AUDIT FIX #4: strict two-value parser
    g_ast_ok  = False
    g_exec_ok = False
    g_correct = False

    if g_fmt_ok and g_sql is not None:
        g_ast_ok = is_ast_safe(g_sql, valid_tables)
        if g_ast_ok:
            ok, r_rows, trunc, _, _ = execute_sandbox_query(db_file, g_sql)
            g_exec_ok = ok and not trunc and r_rows is not None
            if g_exec_ok:
                g_correct = (Counter(r_rows) == Counter(gold_rows))

    # AUDIT FIX #6: greedy_sql_valid = format+AST+execution all passed
    g_sql_valid = g_fmt_ok and g_ast_ok and g_exec_ok

    # ── 2. Majority@8 ────────────────────────────────────────────────────
    # AUDIT FIX #7: maj8_latency_ms measured end-to-end
    # AUDIT FIX #9: cluster_sizes and n_valid_candidates captured
    m_sql, m_rows, m_status, m_cluster_sizes, m_n_valid, m_lat = majority_8_inference(
        row["prompt"], db_file, valid_tables, k=8
    )

    m_correct   = False
    m_exec_ok   = m_rows is not None
    m_sql_valid = m_exec_ok  # Majority@8 only returns a result when it executed
    if m_exec_ok:
        m_correct = (Counter(m_rows) == Counter(gold_rows))

    # AUDIT FIX #9: enriched result record
    results.append({
        "id"                      : row.get("example_id", f"dev_{idx:06d}"),
        "db_id"                   : db_id,
        # Greedy fields
        "greedy_correct"          : g_correct,
        "greedy_sql_valid"        : g_sql_valid,     # format+AST+exec all passed
        "greedy_exec_ok"          : g_exec_ok,
        "greedy_sql"              : g_sql,           # None if parse or AST failed
        "greedy_latency_ms"       : round(g_lat, 3),
        # Majority@8 fields
        "majority8_correct"       : m_correct,
        "majority8_status"        : m_status,
        "majority8_sql_valid"     : m_sql_valid,     # True iff a candidate executed
        "majority8_exec_ok"       : m_exec_ok,
        "majority8_sql"           : m_sql,           # None on ALL_FAILED
        "majority8_cluster_sizes" : m_cluster_sizes, # e.g. [4, 2, 1]
        "majority8_n_valid_candidates": m_n_valid,   # candidates that parsed+executed
        "majority8_latency_ms"    : round(m_lat, 3), # AUDIT FIX #7
        # Provenance
        "adapter_sha256"          : ADAPTER_SHA256,  # AUDIT FIX #1/#9
    })

    if (idx + 1) % 10 == 0:
        cur_g = sum(1 for r in results if r["greedy_correct"]) / len(results) * 100
        cur_m = sum(1 for r in results if r["majority8_correct"]) / len(results) * 100
        print(f"  [{idx+1:4d}/{len(eval_subset)}]  Greedy EX: {cur_g:.1f}%  Maj@8 EX: {cur_m:.1f}%")

# ── Aggregate metrics ─────────────────────────────────────────────────────
total = len(results)

ex_greedy       = sum(1 for r in results if r["greedy_correct"])       / total * 100.0
ex_maj8         = sum(1 for r in results if r["majority8_correct"])     / total * 100.0
valid_greedy_pct= sum(1 for r in results if r["greedy_sql_valid"])      / total * 100.0
valid_maj8_pct  = sum(1 for r in results if r["majority8_sql_valid"])   / total * 100.0
exec_greedy_pct = sum(1 for r in results if r["greedy_exec_ok"])        / total * 100.0
exec_maj8_pct   = sum(1 for r in results if r["majority8_exec_ok"])     / total * 100.0

# AUDIT FIX #8: Percentiles — numpy linear interpolation method (documented)
# np.percentile default method='linear' (Hyndman & Fan Type 7, default since NumPy 1.9)
g_lats  = [r["greedy_latency_ms"]  for r in results]
m_lats  = [r["majority8_latency_ms"] for r in results]
g_p50   = float(np.percentile(g_lats, 50,  method="linear"))
g_p95   = float(np.percentile(g_lats, 95,  method="linear"))
m_p50   = float(np.percentile(m_lats, 50,  method="linear"))
m_p95   = float(np.percentile(m_lats, 95,  method="linear"))

# Decision status breakdown
from collections import Counter as _Counter
status_counts = _Counter(r["majority8_status"] for r in results)

print("\n" + "="*58)
print(" FINAL BENCHMARK SUMMARY")
print("="*58)
print(f"  Eval scope      : {EVAL_MODE} ({total} examples)")
print(f"  GRPO adapter    : {GRPO_ADAPTER.name}")
print(f"  SHA-256         : {ADAPTER_SHA256}")
print("─"*58)
print(f"  Greedy Pass@1")
print(f"    Execution Accuracy (EX)  : {ex_greedy:.2f}%  ({sum(1 for r in results if r['greedy_correct'])}/{total})")
print(f"    SQL Validity (fmt+AST+exec): {valid_greedy_pct:.2f}%")
print(f"    Execution Success         : {exec_greedy_pct:.2f}%")
print(f"    p50 Latency               : {g_p50:.1f} ms")
print(f"    p95 Latency               : {g_p95:.1f} ms")
print("─"*58)
print(f"  Majority@8 (T=0.8, top_p=0.95, k=8)")
print(f"    Execution Accuracy (EX)  : {ex_maj8:.2f}%  ({sum(1 for r in results if r['majority8_correct'])}/{total})")
print(f"    Delta vs Greedy          : {ex_maj8 - ex_greedy:+.2f} pts")
print(f"    SQL Validity (exec'd)    : {valid_maj8_pct:.2f}%")
print(f"    Execution Success        : {exec_maj8_pct:.2f}%")
print(f"    p50 End-to-End Latency   : {m_p50:.1f} ms")
print(f"    p95 End-to-End Latency   : {m_p95:.1f} ms")
print("─"*58)
print("  Decision Status Distribution:")
for status in ["CONSENSUS_ELECTED", "LOW_CONFIDENCE", "NO_CONSENSUS_TIE", "ALL_FAILED"]:
    cnt = status_counts.get(status, 0)
    correct = sum(1 for r in results if r["majority8_status"]==status and r["majority8_correct"])
    acc = (correct / cnt * 100) if cnt else 0.0
    print(f"    {status:<22}: {cnt:3d} ({cnt/total*100:5.1f}%)  Accuracy: {acc:.1f}%")
print("="*58)

# ── Save enriched JSON report ─────────────────────────────────────────────
results_path = REPORTS_DIR / "final_benchmark_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Per-example results saved to: {results_path}")

# ── Save aggregate summary JSON ───────────────────────────────────────────
summary = {
    "eval_mode"             : EVAL_MODE,
    "n_examples"            : total,
    "adapter_path"          : str(GRPO_ADAPTER),
    "adapter_sha256"        : ADAPTER_SHA256,
    "percentile_method"     : "linear (NumPy default, Hyndman & Fan Type 7)",
    "greedy": {
        "execution_accuracy_pct" : round(ex_greedy, 4),
        "sql_validity_pct"       : round(valid_greedy_pct, 4),
        "exec_success_pct"       : round(exec_greedy_pct, 4),
        "p50_latency_ms"         : round(g_p50, 3),
        "p95_latency_ms"         : round(g_p95, 3),
    },
    "majority8": {
        "execution_accuracy_pct" : round(ex_maj8, 4),
        "delta_vs_greedy_pts"    : round(ex_maj8 - ex_greedy, 4),
        "sql_validity_pct"       : round(valid_maj8_pct, 4),
        "exec_success_pct"       : round(exec_maj8_pct, 4),
        "p50_latency_ms"         : round(m_p50, 3),
        "p95_latency_ms"         : round(m_p95, 3),
        "decision_status": {
            s: {
                "count"   : status_counts.get(s, 0),
                "pct"     : round(status_counts.get(s, 0) / total * 100, 2),
                "accuracy": round(
                    sum(1 for r in results if r["majority8_status"]==s and r["majority8_correct"])
                    / max(1, status_counts.get(s, 1)) * 100, 2
                ),
            }
            for s in ["CONSENSUS_ELECTED", "LOW_CONFIDENCE", "NO_CONSENSUS_TIE", "ALL_FAILED"]
        },
    },
}
summary_path = REPORTS_DIR / "benchmark_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Aggregate summary saved to : {summary_path}")

In [ ]:
# ── Cell 7: Mandatory Artifact Zipper ─────────────────────────────────────
import zipfile

output_zip = WORKING_DIR / "sql_grpo_evaluation_results.zip"
print(f"Creating evaluation artifact zip: {output_zip} ...")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(REPORTS_DIR):
        for file in files:
            p = Path(root) / file
            rel = p.relative_to(WORKING_DIR).as_posix()
            zipf.write(p, arcname=rel)

zip_mb = os.path.getsize(output_zip) / (1024 * 1024)
print(f"✅ Created: {output_zip.name} ({zip_mb:.2f} MB)")

from IPython.display import FileLink, display
print("\n👉 Click the link below to download:")
display(FileLink("sql_grpo_evaluation_results.zip"))